# Pose2Text — Colab workflow with Google Drive save

Use the project ZIP and run this notebook on Colab GPU. Checkpoint, results, and a copy of the project ZIP are saved in Google Drive. Once training and test evaluation finish, later Colab sessions reuse those saved files and skip training and dataset download.


## 1. Colab GPU on karo

Colab menu: **Runtime → Change runtime type → T4 GPU** (ya available GPU), phir next cell run karo.


In [ ]:
import os, sys, torch, shutil
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU — select a GPU runtime')
print(shutil.disk_usage('/content'))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PERSIST_ROOT = Path('/content/drive/MyDrive/Pose2Text')
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
print('Persistent Drive folder:', PERSIST_ROOT)


## Google Drive persistence

Drive mount ke baad project ZIP ek baar upload hoga aur Drive mein copy rahega. Checkpoint/metrics/predictions bhi Drive mein save honge, isliye Colab reset hone ke baad completed model dobara train nahi hoga.


In [ ]:
from pathlib import Path
import zipfile, os, shutil
PROJECT_ZIP = Path('/content/Pose2Text.zip')
PROJECT = Path('/content/Pose2Text')
DRIVE_ZIP = PERSIST_ROOT / 'Pose2Text.zip'
if DRIVE_ZIP.exists():
    try:
        with zipfile.ZipFile(DRIVE_ZIP) as archive:
            valid_zip = 'Pose2Text/requirements.txt' in archive.namelist()
    except zipfile.BadZipFile:
        valid_zip = False
    if not valid_zip:
        print('Drive mein purani/invalid project ZIP mili; dobara updated ZIP upload karo.')
        DRIVE_ZIP.unlink()
if not DRIVE_ZIP.exists():
    if PROJECT_ZIP.exists():
        with zipfile.ZipFile(PROJECT_ZIP) as archive:
            if 'Pose2Text/requirements.txt' not in archive.namelist():
                raise FileNotFoundError('Colab Files mein purani ZIP hai. Updated Pose2Text.zip upload karo.')
        shutil.copy2(PROJECT_ZIP, DRIVE_ZIP)
    elif not (PROJECT / 'requirements.txt').exists():
        from google.colab import files
        print('Apne computer se Pose2Text.zip select karke upload karo.')
        uploaded = files.upload()
        choices = [Path('/content') / name for name in uploaded
                   if name.lower().endswith('.zip') and 'pose2text' in name.lower()]
        if not choices:
            raise FileNotFoundError('Pose2Text naam wali ZIP upload nahi hui.')
        with zipfile.ZipFile(choices[0]) as archive:
            if 'Pose2Text/requirements.txt' not in archive.namelist():
                raise FileNotFoundError('Uploaded ZIP purani/invalid hai. Updated Pose2Text.zip chahiye.')
        shutil.copy2(choices[0], DRIVE_ZIP)
if DRIVE_ZIP.exists():
    with zipfile.ZipFile(DRIVE_ZIP) as archive:
        archive.extractall('/content')
if not (PROJECT / 'requirements.txt').is_file():
    raise FileNotFoundError('ZIP mein Pose2Text/requirements.txt nahi mila. Sahi Pose2Text.zip upload karo.')
os.chdir(PROJECT)
print('Project ready:', Path.cwd(), '| Drive copy:', DRIVE_ZIP)


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
CHECKPOINT_DIR = PERSIST_ROOT / 'checkpoints'
RESULTS_DIR = PERSIST_ROOT / 'results'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT = CHECKPOINT_DIR / 'best.pt'
SAVED_METRICS = RESULTS_DIR / 'metrics.json'
HAVE_SAVED_RUN = BEST_CHECKPOINT.is_file() and SAVED_METRICS.is_file()
print('Saved trained model:', BEST_CHECKPOINT.exists())
print('Saved evaluation:', SAVED_METRICS.exists())
print('Will skip dataset download/training:', HAVE_SAVED_RUN)


## Dataset setup (first run only)

Completed checkpoint + evaluation Drive mein milte hi heavy download/extraction skip hota hai. Pehli training ke liye Colab mein lagbhag 45 GB free space chahiye.


In [ ]:
DATA_ROOT = Path('/content/pose2text_data/HumanML3D')
if HAVE_SAVED_RUN:
    print('Saved checkpoint + evaluation found in Drive; skipping dataset download/extraction.')
else:
    from pathlib import Path
    import shutil, subprocess
    old_bad_data = Path('/content/HumanML3D')
    if old_bad_data.exists() and old_bad_data.is_dir() and not old_bad_data.is_symlink():
        # Remove only the exact old failed extraction folder from the earlier workflow.
        if old_bad_data.resolve() == Path('/content/HumanML3D').resolve():
            shutil.rmtree(old_bad_data)
            print('Removed previous failed extraction:', old_bad_data)
    DATA_ARCHIVE = Path('/content/HumanML3D.zip')
    DATA_PARENT = Path('/content/pose2text_data')
    DATA_ROOT = DATA_PARENT / 'HumanML3D'
    DATA_URL = 'https://huggingface.co/AlayaLab/FloodDiffusionDownloads/resolve/main/HumanML3D.zip'
    free_gb = shutil.disk_usage('/content').free / (1024**3)
    print(f'Free Colab disk: {free_gb:.1f} GB')
    if free_gb < 45:
        raise RuntimeError('Disk space kam hai. Colab runtime ke Files panel se purani badi files/folders hatao, phir runtime restart karke dobara try karo.')
    if DATA_ARCHIVE.exists():
        integrity = subprocess.run(['unzip', '-tq', str(DATA_ARCHIVE)], check=False)
        if integrity.returncode != 0:
            print('Existing ZIP is incomplete/corrupt; removing it before a fresh download.')
            DATA_ARCHIVE.unlink()
    if not DATA_ARCHIVE.exists():
        subprocess.run(['wget', '-c', '--show-progress', DATA_URL, '-O', str(DATA_ARCHIVE)], check=True)
    subprocess.run(['unzip', '-tq', str(DATA_ARCHIVE)], check=True)
    DATA_PARENT.mkdir(parents=True, exist_ok=True)
    if not DATA_ROOT.exists():
        subprocess.run(['unzip', '-oq', str(DATA_ARCHIVE), '-d', str(DATA_PARENT)], check=True)
    if not DATA_ROOT.exists():
        candidates = list(DATA_PARENT.rglob('new_joint_vecs'))
        if candidates:
            DATA_ROOT = candidates[0].parent
    print('Dataset root:', DATA_ROOT)
    print('Dataset root entries:', [p.name for p in DATA_ROOT.iterdir()][:20])


## 4. Dataset integrity check

Checker motion arrays, captions, splits aur feature dimensions inspect karega. Agar `EOFError`/corrupt-file error aaye to archive incomplete/corrupt hai: download cell rerun karo. Agar required folders/splits missing hon, extraction root ka structure inspect karo.


In [ ]:
if HAVE_SAVED_RUN:
    print('Saved run found; dataset integrity scan is not needed to view the saved demo/results.')
else:
    assert (DATA_ROOT / 'texts').is_dir(), f'texts folder nahi mila: {DATA_ROOT}'
    assert (DATA_ROOT / 'new_joint_vecs').is_dir(), f'new_joint_vecs folder nahi mila: {DATA_ROOT}'
    for split in ('train.txt', 'val.txt', 'test.txt'):
        assert (DATA_ROOT / split).is_file(), f'{split} nahi mila: {DATA_ROOT}'
    import subprocess, sys
    subprocess.run([sys.executable, 'scripts/check_dataset.py', '--data-root', str(DATA_ROOT)], check=True)


## 5. Real data batch aur model forward pass check

Pehle ek actual HumanML3D sample model mein chala kar tensor shape/loss verify hota hai. Uske baad quick synthetic smoke run bhi alag se hota hai; synthetic result HumanML3D score nahi hai.


In [ ]:
if HAVE_SAVED_RUN:
    print('Saved trained run found; skipping sample load and synthetic smoke run.')
else:
    from src.data.dataset import HumanML3D
    ds = HumanML3D(str(DATA_ROOT))
    train_ids = ds.split_ids('train')
    print('Training IDs:', len(train_ids))
    sample_id = train_ids[0]
    motion = ds.motion(sample_id)
    captions = ds.captions(sample_id)
    print('Sample:', sample_id, '| motion:', motion.shape, '| captions:', len(captions), '| first:', captions[0] if captions else 'none')
    assert motion.ndim == 2 and motion.shape[0] > 0


In [ ]:
if not HAVE_SAVED_RUN:
    import subprocess, sys
    subprocess.run([sys.executable, 'scripts/train.py', '--config', 'configs/smoke_test.yaml', '--smoke-test'], check=True)
    subprocess.run([sys.executable, 'scripts/evaluate.py', '--checkpoint', 'checkpoints/smoke/best.pt', '--results-dir', 'results/smoke'], check=True)
else:
    print('Skipping synthetic smoke run because the saved real-data run is already available.')


## 6. Colab ke liye first real training config

Yeh starter run 2,000 training examples aur 3 epochs se shuru karta hai. Training complete hone ke baad `checkpoints/best.pt` aur validation history save honge. Baad mein `train_limit` ko `null` aur epochs ko zyada karke full training kar sakte ho. Training cell ko dobara chalane par existing best checkpoint overwrite ho sakta hai.


In [ ]:
import yaml
from pathlib import Path
cfg_path = Path('configs/default.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
cfg.update({
    'data_root': str(DATA_ROOT),
    'motion_dir': 'new_joint_vecs',
    'batch_size': 16,
    'epochs': 3,
    'train_limit': 2000,
    'num_workers': 2,
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'results_dir': str(RESULTS_DIR),
})
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg_path.read_text())


In [ ]:
import subprocess, sys
if HAVE_SAVED_RUN:
    print('Drive mein completed checkpoint + results mil gaye. Training skip ki—dobara train nahi hoga.')
elif BEST_CHECKPOINT.exists():
    print('Partial/previous checkpoint found. Training isi checkpoint se resume karega.')
    subprocess.run([sys.executable, 'scripts/train.py', '--config', 'configs/default.yaml', '--resume', str(BEST_CHECKPOINT)], check=True)
else:
    print('First training run. Checkpoint har best epoch par Google Drive mein save hoga.')
    subprocess.run([sys.executable, 'scripts/train.py', '--config', 'configs/default.yaml'], check=True)


## Test evaluation (once)

Evaluation output Drive mein save hota hai. Agli Colab session mein predictions/results seedhe Drive se dikhte hain aur evaluation repeat nahi hota.


In [ ]:
import subprocess, sys
if SAVED_METRICS.exists():
    print('Saved test evaluation already exists in Drive; skipping reevaluation.')
else:
    subprocess.run([sys.executable, 'scripts/evaluate.py', '--checkpoint', str(BEST_CHECKPOINT), '--data-root', str(DATA_ROOT), '--results-dir', str(RESULTS_DIR)], check=True)


In [ ]:
import json
import pandas as pd
for name in ('metrics.json', 'model_comparison.csv', 'validation_history.csv'):
    p = RESULTS_DIR / name
    print('\n###', p, '| exists:', p.exists())
    if p.exists() and p.suffix == '.json': print(json.loads(p.read_text()))
    elif p.exists() and p.suffix == '.csv': display(pd.read_csv(p))
predictions = RESULTS_DIR / 'predictions.csv'
if predictions.exists():
    print('Saved real test-set caption examples:')
    display(pd.read_csv(predictions).head(10))


## Reusable outputs

Checkpoint, metrics, predictions, aur training history Google Drive ke `MyDrive/Pose2Text/` folder mein persist rahenge.


In [ ]:
from pathlib import Path
import zipfile
out_zip = PERSIST_ROOT / 'Pose2Text_outputs.zip'
with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in (RESULTS_DIR, CHECKPOINT_DIR):
        if folder.exists():
            for f in folder.rglob('*'):
                if f.is_file() and f.name != '.gitkeep': z.write(f, f.relative_to(PERSIST_ROOT))
print('Saved permanently in Drive:', out_zip)
print('You only need to train once. Next Colab session will reuse the checkpoint/results above.')
try:
    from google.colab import files
    files.download(str(out_zip))
except ImportError:
    pass


## Demo for your teacher

Saved test predictions, metrics, and validation history display above from Google Drive. They remain available after Colab resets, with no retraining. To launch the interactive app in the original session, set its dataset folder to `/content/pose2text_data/HumanML3D` and checkpoint to `/content/drive/MyDrive/Pose2Text/checkpoints/best.pt`. After a runtime reset, interactive browsing needs the dataset to be downloaded/extracted again; saved captions/results stay in Drive.
